# Clean KTTS Coqui VITS v1
Audits KTTS, rejects measurable audio/transcript defects, and trains a fresh single-speaker end-to-end VITS model. Non-commercial experiment.


In [ ]:
!nvidia-smi
!git clone -q --depth 1 https://github.com/sameerahmad86232/cipher.git /kaggle/working/cipher
!pip install -q 'coqui-tts==0.28.0' 'coqui-tts-trainer>=0.4.0,<0.5.0' pyarrow soundfile scipy
!pip uninstall -y -q tensorboard tensorflow jax jaxlib
!pip install -q --no-cache-dir 'tensorboard==2.19.0'
!python -c "from TTS.tts.models.vits import Vits; from trainer import Trainer; print('Coqui VITS ready')"


In [ ]:
!python /kaggle/working/cipher/scripts/prepare-ktts-matcha.py --repo-dir /kaggle/working/cipher --output /kaggle/working/ktts-vits --local-root /kaggle/input
from pathlib import Path
import json
report=json.loads(Path('/kaggle/working/ktts-vits/preparation-report.json').read_text())
print(report)
assert report['accepted'] >= 2500 and report['hours'] >= 3.5


In [ ]:
import os, subprocess, sys
from pathlib import Path
env=os.environ.copy(); env['CUDA_VISIBLE_DEVICES']='0'; env['PYTHONUNBUFFERED']='1'; env['VITS_EPOCHS']='200'
log=Path('/kaggle/working/vits-training.log')
cmd=[sys.executable,'-u','/kaggle/working/cipher/scripts/train-ktts-coqui-vits.py']
with log.open('w') as handle: result=subprocess.run(cmd,env=env,stdout=handle,stderr=subprocess.STDOUT,text=True,timeout=41400)
print('Training exit code:',result.returncode); print(log.read_text(errors='replace')[-20000:])
if result.returncode != 0: raise RuntimeError('VITS training failed; inspect vits-training.log')


In [ ]:
from pathlib import Path
artifact=Path('/kaggle/working/ktts-vits-clean-v1.zip')
assert artifact.is_file() and artifact.stat().st_size > 1_000_000
print('Artifact:',artifact,'bytes:',artifact.stat().st_size)
